In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from hooks.http_mapbiomas_alerts_hook import HttpMapbiomasAlertsHook

In [2]:
pd.options.display.max_columns = 500

In [3]:

load_dotenv()
hook = HttpMapbiomasAlertsHook(
    username=os.getenv("MAPBIOMAS_USERNAME"),
    password=os.getenv("MAPBIOMAS_PASSWORD"),
)

/Users/pedrohenriqueklein/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
df = pd.read_pickle(f"output/treinamento_modelo_sicar_ac.pkl")

In [ ]:
# dft = df[df.index < 10_000]

In [6]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

In [7]:
MAX_WORKERS = 5

hook._ensure_token()

car_codes = df["cod_imovel"].tolist()
processed = 0
lock = Lock()


def fetch_car(car_code):
    global processed
    try:
        data = hook.get_alerts_by_car(car_code=car_code)
    except Exception as exc:
        print(f"Falha após retries: {car_code} | {exc}")
        data = {"property": None, "alerts": [], "car_code": car_code}

    with lock:
        processed += 1
        if processed % 100 == 0:
            print(f"{processed} processados")
    return data


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    result = list(executor.map(fetch_car, car_codes))

print(f"{processed} processados")

total: 10
total: 20
total: 30
total: 40
total: 50
total: 60
total: 70
total: 80
total: 90
total: 100
total: 110
total: 120
total: 130
total: 140
total: 150
total: 160
total: 170
total: 180
total: 190
total: 200
total: 210
total: 220
total: 230
total: 240
total: 250
total: 260
total: 270
total: 280
total: 290
total: 300
total: 310
total: 320
total: 330
total: 340
total: 350
total: 360
total: 370
total: 380
total: 390
total: 400
total: 410
total: 420
total: 430
total: 440
total: 450
total: 460
total: 470
total: 480
total: 490
total: 500
total: 510
total: 520
total: 530
total: 540
total: 550


KeyboardInterrupt: 

In [8]:
dfz = pd.DataFrame(result)

In [9]:
dfz = dfz.explode("alerts")

In [10]:
dfz

,property,alerts
0,{'propertyCode': 'AC-1200708-19C3C6A0A7B648809...,NaN
1,{'propertyCode': 'AC-1200179-CF396FF9F78E4A0E8...,"{'alertCode': 945418, 'areaHa': 6.9998, 'detec..."
2,{'propertyCode': 'AC-1200138-218DCFB19D064EF38...,NaN
3,{'propertyCode': 'AC-1200500-FB62BF39455A44259...,NaN
4,{'propertyCode': 'AC-1200401-9B67C502E4A945F9B...,"{'alertCode': 197406, 'areaHa': 4.2876, 'detec..."
...,...,...
545,{'propertyCode': 'AC-1200302-B239D7C40BA745EA9...,"{'alertCode': 681269, 'areaHa': 4.026, 'detect..."
546,{'propertyCode': 'AC-1200401-0425716DDF3946D28...,NaN
547,{'propertyCode': 'AC-1200609-E6D9FBED3F6D4923A...,NaN
548,{'propertyCode': 'AC-1200302-EBBDB45E251F4BC8A...,NaN


In [11]:
dfzt = pd.json_normalize(
    data=json.loads(dfz.to_json(orient="records")),
    max_level=None,
    sep="_"
)

In [ ]:
dfzt.to_pickle("output/mapbiomas_alerts_model_ac.pkl")
dfzt.to_csv("output/mapbiomas_alerts_model_ac.txt", index=False, sep=";")

df_parquet = dfzt.copy()
for col in df_parquet.columns:
    sample = df_parquet[col].dropna()
    if df_parquet[col].dtype == "object" and not sample.empty and isinstance(sample.iloc[0], (list, dict)):
        df_parquet[col] = df_parquet[col].apply(
            lambda value: json.dumps(value) if isinstance(value, (list, dict)) else value
        )

df_parquet.to_parquet("output/mapbiomas_alerts_model_ac.parquet", index=False)

In [ ]:
# cerca de 10 alertas por minuto.

In [66]:
dfz[~dfz["alerts"].isna()].shape

(35, 2)